# 13 — Calibrate and compare baseline, control and geometry

Run after both arms of notebook 12 finish. Run all sections in order. This notebook performs inference only and writes reports inside the new experiment directory.

Validation chooses a threshold for each new arm. The baseline retains its original saved threshold. Each true positive requires correct class and box IoU >=0.50. A negative image with any accepted detection is a false positive.

Predefined goals: at least 90% recall for each class and at most 5% negative false positives. A useful geometry candidate must also reduce validation false positives relative to BOTH baseline and control, and lose no more than one percentage point of recall versus baseline in either class. This tests whether geometry helps beyond additional training. The comparison is exploratory and a single seed is not statistical proof.

Validation decisions are saved before the frozen-threshold development-test evaluation. No threshold/checkpoint is selected using test or QA results. These test images have been inspected previously and contain correlated video frames; this is not a fresh blind benchmark. No model is promoted or exported.

In [1]:
from pathlib import Path
from collections import Counter
import sys
import json
import hashlib
import time
import random
import html

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, HTML

PROJECT = Path("/home/joe/PycharmProjects/id-capture")
SOURCE = PROJECT / "training/vendor/YOLOX"
for directory in (str(PROJECT), str(SOURCE)):
    if directory not in sys.path:
        sys.path.insert(0, directory)
from training.yolox_workflow import (
    make_model, predict_records, summarize, audit, save_json,
    letterbox, show_progress, NAMES,
)

def sha256_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def table(rows):
    if not rows:
        print("No rows.")
        return
    columns = list(rows[0])
    head = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(row[c]))}</td>" for c in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

ROOT = PROJECT / "training/runs/yolox_geometry_experiments"
experiment = json.loads((ROOT / "latest_completed.json").read_text())
EXPERIMENT = Path(experiment["experiment"])
MANIFEST_PATH = Path(experiment["manifest"])
assert sha256_file(MANIFEST_PATH) == experiment["manifest_sha256"], "Dataset changed."
manifest = json.loads(MANIFEST_PATH.read_text())
audit(manifest["records"])
CHECKPOINTS = {name: Path(path) for name,path in experiment["checkpoints"].items()}
assert all(path.is_file() for path in CHECKPOINTS.values())
assert sha256_file(CHECKPOINTS["Baseline"]) == experiment["baseline_sha256"]
SIZE, BATCH = experiment["size"], 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)

MIN_RECALL, MAX_NEGATIVE_FP, MAX_RECALL_DROP = 0.90, 0.05, 0.01
baseline_calibration = json.loads((CHECKPOINTS["Baseline"].parent / "calibration.json").read_text())
assert baseline_calibration["checkpoint"] == str(CHECKPOINTS["Baseline"])
BASELINE_THRESHOLD = float(baseline_calibration["threshold"])
assert 0.05 <= BASELINE_THRESHOLD <= 1.0
THRESHOLDS = sorted(set([0.25] + [round(float(t),2) for t in np.arange(0.10,0.96,0.05)]))
print("Frozen baseline threshold:", BASELINE_THRESHOLD)


train 13100 {('midv500', 'none'): 4292, ('midv500', 'driving_licence'): 2362, ('midv500', 'passport'): 2346, ('midv2019', 'none'): 1680, ('midv2019', 'driving_licence'): 960, ('midv2019', 'passport'): 960, ('midv2020', 'none'): 200, ('midv2020', 'passport'): 300}
val 4400 {('midv500', 'none'): 1511, ('midv500', 'passport'): 890, ('midv500', 'driving_licence'): 599, ('midv2019', 'none'): 600, ('midv2019', 'passport'): 360, ('midv2019', 'driving_licence'): 240, ('midv2020', 'none'): 100, ('midv2020', 'passport'): 100}
test 4500 {('midv500', 'passport'): 884, ('midv500', 'none'): 1537, ('midv500', 'driving_licence'): 579, ('midv2019', 'passport'): 360, ('midv2019', 'driving_licence'): 240, ('midv2019', 'none'): 600, ('midv2020', 'none'): 300}
PASS: no document design crosses splits; all images exist.
Experiment: /home/joe/PycharmProjects/id-capture/training/runs/yolox_geometry_experiments/20261022_170347_473302
Device: cuda
Frozen baseline threshold: 0.2


## Validation calibration and source-level metrics
Only validation records are used here. Inference is cached in memory one model at a time; thresholds are evaluated from the same cached predictions.

In [2]:
validation = [r for r in manifest["records"] if r["split"] == "val"]
calibrations, validation_rows = {}, []
for name, checkpoint in CHECKPOINTS.items():
    model = make_model(checkpoint).to(DEVICE).eval()
    predictions = predict_records(model, validation, SIZE, BATCH, minimum_confidence=0.05)
    sweep = [
        summarize(validation, predictions, threshold)[0]
        for threshold in (THRESHOLDS if name != "Baseline" else [BASELINE_THRESHOLD])
    ]
    eligible = [
        row for row in sweep
        if row["licence_recall"] >= MIN_RECALL and row["passport_recall"] >= MIN_RECALL
    ]
    if name == "Baseline":
        choice = sweep[0]
    elif eligible:
        choice = min(eligible, key=lambda row: (
            row["negative_fp"], -min(row["licence_recall"],row["passport_recall"]),
            row["threshold"],
        ))
    else:
        choice = next(row for row in sweep if row["threshold"] == 0.25)
    rows = summarize(validation, predictions, choice["threshold"])
    calibration = dict(
        model=name, checkpoint=str(checkpoint), checkpoint_sha256=sha256_file(checkpoint),
        manifest_sha256=experiment["manifest_sha256"], threshold=choice["threshold"],
        recall_target_met=bool(eligible),
        all_targets_met=bool(eligible) and choice["negative_fp"] <= MAX_NEGATIVE_FP,
        sweep=sweep, rows=rows,
    )
    calibrations[name] = calibration
    validation_rows.extend(dict(model=name, **row) for row in rows)
    save_json(EXPERIMENT / f"{name.lower()}_calibration.json", calibration)
    print(name, "| selected confidence:", choice["threshold"],
          "| recall targets:", calibration["recall_target_met"],
          "| all targets:", calibration["all_targets_met"])
    if name != "Baseline":
        table(sweep)
    del predictions, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
table(validation_rows)


Loaded pretrained tensors: 462 | New class-head tensors: 0
Inference 16/4400 | 0.4% | Elapsed 0:00:01 | Remaining 0:05:45 | Estimated finish 2026-10-22 18:26:39 BST
Inference 816/4400 | 18.5% | Elapsed 0:00:05 | Remaining 0:00:25 | Estimated finish 2026-10-22 18:21:24 BST
Inference 1616/4400 | 36.7% | Elapsed 0:00:10 | Remaining 0:00:17 | Estimated finish 2026-10-22 18:21:21 BST
Inference 2416/4400 | 54.9% | Elapsed 0:00:14 | Remaining 0:00:12 | Estimated finish 2026-10-22 18:21:19 BST
Inference 3216/4400 | 73.1% | Elapsed 0:00:18 | Remaining 0:00:06 | Estimated finish 2026-10-22 18:21:18 BST
Inference 4016/4400 | 91.3% | Elapsed 0:00:22 | Remaining 0:00:02 | Estimated finish 2026-10-22 18:21:17 BST
Inference 4400/4400 | 100.0% | Elapsed 0:00:24 | Remaining 0:00:00 | Finished 2026-10-22 18:21:17 BST
Baseline | selected confidence: 0.2 | recall targets: True | all targets: False
Loaded pretrained tensors: 462 | New class-head tensors: 0
Inference 16/4400 | 0.4% | Elapsed 0:00:00 | Remai

source,threshold,images,licence_count,passport_count,negative_count,licence_recall,passport_recall,negative_fp
combined,0.1,4400,839,1350,2211,0.9690107270560191,0.9881481481481481,0.20307553143374038
combined,0.15,4400,839,1350,2211,0.966626936829559,0.9874074074074074,0.19674355495251017
combined,0.2,4400,839,1350,2211,0.964243146603099,0.9874074074074074,0.19086386250565354
combined,0.25,4400,839,1350,2211,0.9618593563766389,0.9874074074074074,0.1890547263681592
combined,0.3,4400,839,1350,2211,0.9606674612634089,0.9874074074074074,0.18769787426503845
combined,0.35,4400,839,1350,2211,0.9582836710369488,0.9859259259259259,0.18453188602442333
combined,0.4,4400,839,1350,2211,0.9570917759237187,0.9859259259259259,0.18091361374943465
combined,0.45,4400,839,1350,2211,0.9558998808104887,0.9859259259259259,0.1786521935775667
combined,0.5,4400,839,1350,2211,0.9535160905840286,0.9859259259259259,0.17503392130257803
combined,0.55,4400,839,1350,2211,0.9535160905840286,0.9859259259259259,0.17141564902758932


Loaded pretrained tensors: 462 | New class-head tensors: 0
Inference 16/4400 | 0.4% | Elapsed 0:00:00 | Remaining 0:00:23 | Estimated finish 2026-10-22 18:22:07 BST
Inference 816/4400 | 18.5% | Elapsed 0:00:05 | Remaining 0:00:22 | Estimated finish 2026-10-22 18:22:10 BST
Inference 1616/4400 | 36.7% | Elapsed 0:00:09 | Remaining 0:00:16 | Estimated finish 2026-10-22 18:22:09 BST
Inference 2416/4400 | 54.9% | Elapsed 0:00:13 | Remaining 0:00:11 | Estimated finish 2026-10-22 18:22:07 BST
Inference 3216/4400 | 73.1% | Elapsed 0:00:17 | Remaining 0:00:06 | Estimated finish 2026-10-22 18:22:07 BST
Inference 4016/4400 | 91.3% | Elapsed 0:00:21 | Remaining 0:00:02 | Estimated finish 2026-10-22 18:22:06 BST
Inference 4400/4400 | 100.0% | Elapsed 0:00:23 | Remaining 0:00:00 | Finished 2026-10-22 18:22:06 BST
Geometry | selected confidence: 0.8 | recall targets: True | all targets: False


source,threshold,images,licence_count,passport_count,negative_count,licence_recall,passport_recall,negative_fp
combined,0.1,4400,839,1350,2211,0.9773539928486293,0.9851851851851852,0.13930348258706468
combined,0.15,4400,839,1350,2211,0.9702026221692491,0.9814814814814815,0.12663952962460426
combined,0.2,4400,839,1350,2211,0.9702026221692491,0.9807407407407407,0.11714156490275893
combined,0.25,4400,839,1350,2211,0.9678188319427891,0.98,0.10990502035278155
combined,0.3,4400,839,1350,2211,0.966626936829559,0.9785185185185186,0.10402532790592492
combined,0.35,4400,839,1350,2211,0.9618593563766389,0.9785185185185186,0.1013116236996834
combined,0.4,4400,839,1350,2211,0.9606674612634089,0.9785185185185186,0.09407507914970602
combined,0.45,4400,839,1350,2211,0.9570917759237187,0.9777777777777777,0.0877431026684758
combined,0.5,4400,839,1350,2211,0.9570917759237187,0.977037037037037,0.0845771144278607
combined,0.55,4400,839,1350,2211,0.9558998808104887,0.977037037037037,0.0814111261872456


model,source,threshold,images,licence_count,passport_count,negative_count,licence_recall,passport_recall,negative_fp
Baseline,combined,0.2,4400,839,1350,2211,0.902264600715137,0.9881481481481481,0.17503392130257803
Baseline,midv2019,0.2,1200,240,360,600,0.8541666666666666,1.0,0.255
Baseline,midv2020,0.2,200,0,100,100,None,0.97,0.3
Baseline,midv500,0.2,3000,599,890,1511,0.9215358931552587,0.9853932584269663,0.13500992720052946
Control,combined,0.95,4400,839,1350,2211,0.9106078665077473,0.9762962962962963,0.1216644052464948
Control,midv2019,0.95,1200,240,360,600,0.8208333333333333,0.9944444444444445,0.17333333333333334
Control,midv2020,0.95,200,0,100,100,None,0.97,0.14
Control,midv500,0.95,3000,599,890,1511,0.9465776293823038,0.9696629213483146,0.099933818663137
Geometry,combined,0.8,4400,839,1350,2211,0.930870083432658,0.9607407407407408,0.052917232021709636
Geometry,midv2019,0.8,1200,240,360,600,0.8833333333333333,0.9888888888888889,0.08333333333333333


## Save the validation decision before inspecting test
A candidate can be an improvement without meeting the final 5% target. These are recorded separately. Failure does not trigger automatic threshold changes or another training run.

In [3]:
combined = {
    name: next(row for row in calibration["rows"] if row["source"] == "combined")
    for name, calibration in calibrations.items()
}
base, control, geo = (combined[name] for name in ("Baseline","Control","Geometry"))
recall_ok = (
    geo["licence_recall"] >= MIN_RECALL and geo["passport_recall"] >= MIN_RECALL
    and geo["licence_recall"] >= base["licence_recall"] - MAX_RECALL_DROP
    and geo["passport_recall"] >= base["passport_recall"] - MAX_RECALL_DROP
)
geometry_helped = (
    recall_ok and geo["negative_fp"] < base["negative_fp"]
    and geo["negative_fp"] < control["negative_fp"]
)
validation_decision = dict(
    selection_split="val", minimum_recall=MIN_RECALL,
    maximum_negative_fp=MAX_NEGATIVE_FP, maximum_recall_drop=MAX_RECALL_DROP,
    geometry_improves_over_baseline_and_control=bool(geometry_helped),
    geometry_meets_all_targets=bool(recall_ok and geo["negative_fp"] <= MAX_NEGATIVE_FP),
    metrics=combined,
    thresholds={name: cal["threshold"] for name,cal in calibrations.items()},
    checkpoint_hashes={name: cal["checkpoint_sha256"] for name,cal in calibrations.items()},
    manifest_sha256=experiment["manifest_sha256"],
)
save_json(EXPERIMENT / "validation_decision.json", validation_decision)
table([dict(model=name, **row) for name,row in combined.items()])
print("Geometry improves over baseline and control:", geometry_helped)
print("Geometry meets all targets:", validation_decision["geometry_meets_all_targets"])
print("Decision frozen before test. Continue for a development-test comparison, not selection.")


model,source,threshold,images,licence_count,passport_count,negative_count,licence_recall,passport_recall,negative_fp
Baseline,combined,0.2,4400,839,1350,2211,0.902264600715137,0.9881481481481481,0.17503392130257803
Control,combined,0.95,4400,839,1350,2211,0.9106078665077473,0.9762962962962963,0.1216644052464948
Geometry,combined,0.8,4400,839,1350,2211,0.930870083432658,0.9607407407407408,0.052917232021709636


Geometry improves over baseline and control: False
Geometry meets all targets: False
Decision frozen before test. Continue for a development-test comparison, not selection.


## Frozen-threshold development-test comparison
All three models use their validation-selected thresholds. Report by source as well as overall. MIDV-2020 in this split has only negatives; its target recalls are N/A. Do not retune based on these results.

In [4]:
test_records = [r for r in manifest["records"] if r["split"] == "test"]
test_rows = []
for name, checkpoint in CHECKPOINTS.items():
    calibration = json.loads((EXPERIMENT / f"{name.lower()}_calibration.json").read_text())
    assert sha256_file(checkpoint) == calibration["checkpoint_sha256"]
    model = make_model(checkpoint).to(DEVICE).eval()
    predictions = predict_records(model, test_records, SIZE, BATCH, minimum_confidence=0.05)
    rows = summarize(test_records, predictions, calibration["threshold"])
    test_rows.extend(dict(model=name, **row) for row in rows)
    del predictions, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
table(test_rows)
save_json(EXPERIMENT / "development_test_comparison.json", dict(
    split="test", rows=test_rows, validation_decision=validation_decision,
    limitation="Previously inspected development test with correlated frames; not used for selection.",
))
print("Reports:", EXPERIMENT)
print("Next: notebook 14 for phone photos and the 12 saved negative QA fixtures.")
print("Baseline pointers and app ONNX remain unchanged.")


Loaded pretrained tensors: 462 | New class-head tensors: 0
Inference 16/4500 | 0.4% | Elapsed 0:00:00 | Remaining 0:00:28 | Estimated finish 2026-10-22 18:22:38 BST
Inference 816/4500 | 18.1% | Elapsed 0:00:04 | Remaining 0:00:19 | Estimated finish 2026-10-22 18:22:33 BST
Inference 1616/4500 | 35.9% | Elapsed 0:00:09 | Remaining 0:00:16 | Estimated finish 2026-10-22 18:22:36 BST
Inference 2416/4500 | 53.7% | Elapsed 0:00:14 | Remaining 0:00:12 | Estimated finish 2026-10-22 18:22:36 BST
Inference 3216/4500 | 71.5% | Elapsed 0:00:19 | Remaining 0:00:07 | Estimated finish 2026-10-22 18:22:37 BST
Inference 4016/4500 | 89.2% | Elapsed 0:00:23 | Remaining 0:00:02 | Estimated finish 2026-10-22 18:22:36 BST
Inference 4500/4500 | 100.0% | Elapsed 0:00:26 | Remaining 0:00:00 | Finished 2026-10-22 18:22:36 BST
Loaded pretrained tensors: 462 | New class-head tensors: 0
Inference 16/4500 | 0.4% | Elapsed 0:00:00 | Remaining 0:00:25 | Estimated finish 2026-10-22 18:23:02 BST
Inference 816/4500 | 18.

model,source,threshold,images,licence_count,passport_count,negative_count,licence_recall,passport_recall,negative_fp
Baseline,combined,0.2,4500,819,1244,2437,0.945054945054945,0.9919614147909968,0.08001641362330734
Baseline,midv2019,0.2,1200,240,360,600,0.9708333333333333,1.0,0.06833333333333333
Baseline,midv2020,0.2,300,0,0,300,None,None,0.03333333333333333
Baseline,midv500,0.2,3000,579,884,1537,0.9343696027633851,0.9886877828054299,0.0936890045543266
Control,combined,0.95,4500,819,1244,2437,0.9584859584859585,0.977491961414791,0.09273697168649979
Control,midv2019,0.95,1200,240,360,600,0.9833333333333333,0.9944444444444445,0.07666666666666666
Control,midv2020,0.95,300,0,0,300,None,None,0.02
Control,midv500,0.95,3000,579,884,1537,0.9481865284974094,0.9705882352941176,0.11320754716981132
Geometry,combined,0.8,4500,819,1244,2437,0.9242979242979243,0.97508038585209,0.022979072630283134
Geometry,midv2019,0.8,1200,240,360,600,0.9333333333333333,0.9805555555555555,0.021666666666666667


Reports: /home/joe/PycharmProjects/id-capture/training/runs/yolox_geometry_experiments/20261022_170347_473302
Next: notebook 14 for phone photos and the 12 saved negative QA fixtures.
Baseline pointers and app ONNX remain unchanged.
